# Comparing Across Archives

**Computational Text Analysis Pack · Module 04 — Comparing Corpora**

*Technique:* Keyness between unlike corpora; confounds you cannot remove

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/03-comparing-corpora/04c_comparing_across_archives.ipynb)

Data: `colonial_texts.csv`, `masakhanews.csv`, `african_legal.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv', 'masakhanews.csv', 'african_legal.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
df       = pd.read_csv('masakhanews.csv')
df_legal = pd.read_csv('african_legal.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

The comparisons so far were fair: same corpus, same language, different topic.
This one is not fair, and that is the lesson.

Comparing pre-1929 European writing about Africa with contemporary African
journalism is a comparison across **period, genre, purpose, audience and
authorship at once**. The keyness list will be striking. Attributing it to any
one of those five is the error this notebook exists to prevent.

## The tooling

**`collections.Counter`** and plain arithmetic. Keyness needs no library: the
log-likelihood formula is four lines. **quanteda** (R) and **Sketch Engine** are
the field's usual implementations of exactly this.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
from collections import Counter

def frequencies(texts):
    """Token counts across a set of documents."""
    counts = Counter()
    for t in texts:
        counts.update(tokenise(t))
    return counts

def freq_frame(counts, top=25):
    total = sum(counts.values())
    rows = [{"rank": i, "word": w, "count": n, "per_1000": n / total * 1000}
            for i, (w, n) in enumerate(counts.most_common(top), start=1)]
    return pd.DataFrame(rows)

In [ ]:
def log_likelihood(counts_a, counts_b, min_count=10):
    """Words that distinguish corpus A from corpus B.

    The G-squared statistic corpus linguists call *keyness*. A word scores high
    when it is far more common in one corpus than its overall rate predicts.
    Positive `diff` means over-used in A.
    """
    na, nb = sum(counts_a.values()), sum(counts_b.values())
    rows = []
    for w in set(counts_a) | set(counts_b):
        o1, o2 = counts_a.get(w, 0), counts_b.get(w, 0)
        if o1 + o2 < min_count:
            continue
        e1 = na * (o1 + o2) / (na + nb)
        e2 = nb * (o1 + o2) / (na + nb)
        g2 = 2 * ((o1 * np.log(o1 / e1) if o1 else 0)
                  + (o2 * np.log(o2 / e2) if o2 else 0))
        rows.append({"word": w, "count_a": o1, "count_b": o2,
                     "per_1000_a": o1 / na * 1000, "per_1000_b": o2 / nb * 1000,
                     "keyness": g2, "over_used_in": "A" if o1 / na > o2 / nb else "B"})
    out = pd.DataFrame(rows).sort_values("keyness", ascending=False)
    return out.reset_index(drop=True)

In [ ]:
colonial = frequencies(df_colonial["text"])
news = frequencies(df[df["language"] == "English"]["text"])
legal = frequencies(df_legal["text"])

for name, c in [("colonial-era writing", colonial),
                ("contemporary news (English)", news),
                ("court & governance texts", legal)]:
    print(f"{name:<32}{sum(c.values()):>10,} tokens   {len(c):>8,} types")

In [ ]:
k = log_likelihood(colonial, news, min_count=50)
print("Distinctive of pre-1929 writing about Africa:")
print(k[k["over_used_in"] == "A"].head(18)[
      ["word", "per_1000_a", "per_1000_b", "keyness"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print()
print("Distinctive of contemporary African journalism:")
print(k[k["over_used_in"] == "B"].head(18)[
      ["word", "per_1000_a", "per_1000_b", "keyness"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

## What you are actually seeing

Three different things are mixed into that list and cannot be separated:

| What differs | Words it produces |
|---|---|
| **Period** | *thee*, *whilst*, *upon* — archaic English, nothing to do with Africa |
| **Genre** | narrative prose vs news writing — *said*, *reported*, *told* |
| **Stance** | *native*, *savage*, *tribe* — the finding you were looking for |

The third is the interesting one and it is buried in the other two. Isolating it
takes judgement, not a better statistic.

In [ ]:
# One partial control: restrict to a vocabulary that exists in both eras.
# It does not remove the confound — it narrows it.
TERMS = ["africa", "african", "native", "tribe", "chief", "government",
         "people", "country", "land", "law", "trade", "school", "women", "war"]

na, nb = sum(colonial.values()), sum(news.values())
rows = []
for t in TERMS:
    rows.append({"word": t,
                 "colonial_per_1k": colonial.get(t, 0) / na * 1000,
                 "news_per_1k": news.get(t, 0) / nb * 1000})
comp = pd.DataFrame(rows)
comp["log_ratio"] = np.log2(comp["colonial_per_1k"].clip(lower=1e-4)
                            / comp["news_per_1k"].clip(lower=1e-4))
print(comp.sort_values("log_ratio", ascending=False)
          .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
c = comp.sort_values("log_ratio")
ax.barh(c["word"], c["log_ratio"],
        color=["steelblue" if v < 0 else "indianred" for v in c["log_ratio"]])
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("← more common in contemporary news    |    more common pre-1929 →")
ax.set_title("A shared vocabulary, two archives (log₂ ratio)")
plt.tight_layout(); plt.show()

### 🔧 Your turn 1

Swap `news` for `legal` — the court and governance corpus — and re-run.

The legal corpus is also formal, also institutional, also partly archival. Does
*native* behave differently against this reference than against journalism? A
result that survives a change of reference is much stronger than one that does
not.

## The write-up that survives review

Compare these two sentences. Only one is defensible from what you just ran.

> ✗ *"Colonial writing dehumanised Africans, using 'native' 14 times more often
> than African writers do."*

> ✓ *"In this corpus of pre-1929 English-language writing about Africa, 'native'
> occurs at 1.8 per 1,000 words against 0.13 in contemporary African news
> reporting (log ratio 3.8). The two corpora differ in period, genre and
> authorship, so the contrast identifies a vocabulary characteristic of the
> colonial archive rather than isolating any one of those factors."*

The second names the corpora, gives rates and an effect size, and states the
confound. It takes one extra sentence and it is the difference between a claim a
reviewer accepts and one they reject.

### 🔧 Your turn 2

Write the limits paragraph for your own comparison, before you write the finding.

Name: the two corpora, what differs between them besides the thing you care
about, and one alternative explanation you cannot rule out.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** *Native* is heavily over-used in the colonial corpus against
both references, which strengthens the reading — it is not an artefact of
comparing against journalism specifically. Institutional vocabulary (*court*,
*law*, *government*) behaves differently: it is common in the legal corpus by
genre, so those words tell you about document type rather than period. Watching
which words move when the reference changes, and which do not, is the most
useful diagnostic in this notebook.

**Your turn 2.** A strong limits paragraph for the colonial/news comparison names
five confounds — period, genre, purpose, audience, authorship — and concedes that
the archaic-English words in the keyness list prove period alone is doing real
work. The alternative explanation you cannot rule out is that any two corpora a
century apart differ this much, regardless of subject. The way to test it would
be a third corpus: pre-1929 English writing *not* about Africa. That comparison
is available to you and would make a strong paper.

</details>

## Where this stops

You can compare corpora and describe the difference honestly. Next: **Module 05 —
concordance and collocation**, which stops counting words in isolation and starts
looking at the company they keep.